In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

contacts = pd.read_csv("../data/contacts.csv")
opportunities = pd.read_csv("../data/opportunities.csv")
campaigns = pd.read_csv("../data/campaigns.csv")

contacts.columns = contacts.columns.str.lower().str.replace(" ", "_")
opportunities.columns = opportunities.columns.str.lower().str.replace(" ", "_")
campaigns.columns = campaigns.columns.str.lower().str.replace(" ", "_")

print("Contacts:", len(contacts))
print("Opportunities:", len(opportunities))
print("Campaigns:", len(campaigns))

Contacts: 1277
Opportunities: 192
Campaigns: 18


In [2]:
campaigns = campaigns.rename(columns={
    'amount_spent_(usd)': 'amount_spent_usd',
    'cost_per_results': 'cost_per_result',
    'cpm_(cost_per_1,000_impressions)_(usd)': 'cpm_usd',
    'cpc_(cost_per_link_click)_(usd)': 'cpc_link_click_usd',
    'ctr_(link_click-through_rate)': 'ctr_link_click'
})

opportunities['status'] = opportunities['status'].astype(str).str.strip().str.lower()

print(campaigns.columns.tolist())

['reporting_starts', 'reporting_ends', 'campaign_name', 'campaign_delivery', 'attribution_setting', 'results', 'result_indicator', 'reach', 'frequency', 'cost_per_result', 'ad_set_budget', 'ad_set_budget_type', 'amount_spent_usd', 'ends', 'impressions', 'cpm_usd', 'link_clicks', 'shop_clicks', 'cpc_link_click_usd', 'ctr_link_click', 'clicks_(all)', 'ctr_(all)', 'cpc_(all)_(usd)', 'landing_page_views', 'cost_per_landing_page_view_(usd)']


In [3]:
total_contacts = len(contacts)
total_opportunities = len(opportunities)
total_campaigns = len(campaigns)

total_spend = campaigns['amount_spent_usd'].sum()
total_results = campaigns['results'].sum()
total_lead_value = opportunities['lead_value'].sum()

cost_per_result = total_spend / total_results
opportunity_rate = total_opportunities / total_contacts
estimated_roas = total_lead_value / total_spend

print("Total Contacts:", total_contacts)
print("Total Opportunities:", total_opportunities)
print("Total Campaigns:", total_campaigns)
print("Total Ad Spend:", round(total_spend, 2))
print("Total Results:", total_results)
print("Cost Per Result:", round(cost_per_result, 2))
print("Opportunity Rate:", round(opportunity_rate * 100, 2), "%")
print("Total Lead Value:", round(total_lead_value, 2))
print("Estimated ROAS:", round(estimated_roas, 2))

Total Contacts: 1277
Total Opportunities: 192
Total Campaigns: 18
Total Ad Spend: 22019.11
Total Results: 327.0
Cost Per Result: 67.34
Opportunity Rate: 15.04 %
Total Lead Value: 231900
Estimated ROAS: 10.53


In [4]:
conn = sqlite3.connect("../database/business_analytics.db")

contacts.to_sql("contacts", conn, if_exists="replace", index=False)
opportunities.to_sql("opportunities", conn, if_exists="replace", index=False)
campaigns.to_sql("campaigns", conn, if_exists="replace", index=False)

conn.close()

print("Database created successfully.")

Database created successfully.


In [5]:
import pandas as pd
import sqlite3
from pathlib import Path

# Paths
BASE_DIR = Path("..")
DB_PATH = BASE_DIR / "database" / "business_analytics.db"
DATA_DIR = BASE_DIR / "data"

# Connect to database
conn = sqlite3.connect(DB_PATH)

# QUERY 1 — TOP PERFORMING CAMPAIGNS
top_campaigns_sql = pd.read_sql_query("""
SELECT
    campaign_name,
    SUM(amount_spent_usd) AS total_spend,
    SUM(results) AS total_results,
    ROUND(AVG(cost_per_result), 2) AS avg_cost_per_result
FROM campaigns
GROUP BY campaign_name
ORDER BY total_results DESC;
""", conn)

print("Top Performing Campaigns")
print(top_campaigns_sql.head(10))

top_campaigns_sql.to_csv(DATA_DIR / "top_campaigns_sql.csv", index=False)


# QUERY 2 — PIPELINE DISTRIBUTION
pipeline_distribution_sql = pd.read_sql_query("""
SELECT
    stage,
    COUNT(*) AS total_opportunities
FROM opportunities
GROUP BY stage
ORDER BY total_opportunities DESC;
""", conn)

print("\nPipeline Distribution")
print(pipeline_distribution_sql)

pipeline_distribution_sql.to_csv(DATA_DIR / "pipeline_distribution_sql.csv", index=False)


# QUERY 3 — REVENUE BY STAGE
revenue_by_stage_sql = pd.read_sql_query("""
SELECT
    stage,
    SUM(lead_value) AS total_lead_value
FROM opportunities
GROUP BY stage
ORDER BY total_lead_value DESC;
""", conn)

print("\nRevenue by Pipeline Stage")
print(revenue_by_stage_sql)

revenue_by_stage_sql.to_csv(DATA_DIR / "revenue_by_stage_sql.csv", index=False)


# QUERY 4 — LEAD SOURCES
lead_sources_sql = pd.read_sql_query("""
SELECT
    source,
    COUNT(*) AS total_leads
FROM opportunities
GROUP BY source
ORDER BY total_leads DESC;
""", conn)

print("\nLead Sources")
print(lead_sources_sql)

lead_sources_sql.to_csv(DATA_DIR / "lead_sources_sql.csv", index=False)

conn.close()

print("\nAll SQL result CSV files were saved successfully.")

Top Performing Campaigns
                               campaign_name  total_spend  total_results  \
0       New Leads Campaign kitchens - Copy 2     11062.60          177.0   
1          New Leads Campaign impact windows      9974.48          139.0   
2                New Roofing Campaign - Copy       982.03           11.0   
3  ‏‎New Leads Campaign impact windows‎‏ - 2         0.00            NaN   
4           ‏‎New Leads Campaign Roofs‎‏ - 2         0.00            NaN   
5                       New Roofing Campaign         0.00            NaN   
6            New Leads Campaign paint - Copy         0.00            NaN   
7                   New Leads Campaign paint         0.00            NaN   
8     New Leads Campaign impact windows - V2         0.00            NaN   
9         New Leads Campaign Roofs V2 - Copy         0.00            NaN   

   avg_cost_per_result  
0                62.50  
1                71.76  
2                89.28  
3                  NaN  
4            